# NoSQL Databases

## Why NoSQL?

### CAP Theorem

A distributed system can guarantee at most **2 of 3** properties:

$$\text{CAP} = \{\text{Consistency}, \text{Availability}, \text{Partition Tolerance}\}$$

- **Consistency (C)**: Every read sees the most recent write
- **Availability (A)**: Every request gets a response (not necessarily latest data)
- **Partition Tolerance (P)**: System continues despite network failures

| Database | Guarantees | Trade-off |
|----------|-----------|----------|
| PostgreSQL, MySQL | CP | Not highly available during partition |
| Cassandra, DynamoDB | AP | Eventually consistent |
| MongoDB | CP (configurable) | Can trade C for A |
| Redis | CP | In-memory, fast |

## NoSQL Types

| Type | Examples | Best For |
|------|---------|----------|
| **Document** | MongoDB, CouchDB, Firestore | JSON-like records, flexible schema |
| **Key-Value** | Redis, DynamoDB, Memcached | Caching, sessions, leaderboards |
| **Column-Family** | Cassandra, HBase | Time-series, analytics, wide tables |
| **Graph** | Neo4j, Amazon Neptune | Social networks, recommendations |
| **Search** | Elasticsearch, OpenSearch | Full-text search, logs |
| **Vector** | Pinecone, Weaviate, Qdrant | ML embeddings, similarity search |

## MongoDB

MongoDB stores data as **BSON** (Binary JSON) documents in **collections** (like tables).

```json
// Document example
{
  "_id": ObjectId("507f1f77bcf86cd799439011"),
  "username": "alice",
  "email": "alice@example.com",
  "tags": ["ml", "python", "fastapi"],
  "address": {
    "city": "New York",
    "zip": "10001"
  },
  "created_at": ISODate("2024-01-15")
}
```

In [1]:
# pip install pymongo motor (motor = async pymongo)
# Start MongoDB: docker run -d -p 27017:27017 mongo

# Simulating MongoDB operations with Python dicts
# In real use, replace with: from pymongo import MongoClient

print("MongoDB CRUD Operations Examples")
print("=" * 50)

mongo_operations = '''
# Connect
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
db = client["ai_learning"]
users = db["users"]

# CREATE insertOne / insertMany
users.insert_one({"username": "alice", "age": 30, "tags": ["ml", "python"]})
users.insert_many([
    {"username": "bob", "age": 25},
    {"username": "carol", "age": 35, "tags": ["data"]}
])

# READ find / findOne
user = users.find_one({"username": "alice"})
all_users = list(users.find({"age": {"$gt": 25}}))
projected = list(users.find({}, {"username": 1, "_id": 0}))  # projection
sorted_users = list(users.find().sort("age", -1).limit(5))

# Query Operators:
# Comparison: $eq, $ne, $gt, $gte, $lt, $lte, $in, $nin
# Logical:    $and, $or, $nor, $not
# Element:    $exists, $type
# Array:      $all, $elemMatch, $size

# Complex query
result = users.find({
    "$and": [
        {"age": {"$gte": 25, "$lte": 40}},
        {"tags": {"$in": ["ml", "python"]}},
    ]
})

# UPDATE
users.update_one(
    {"username": "alice"},
    {"$set": {"age": 31}, "$push": {"tags": "fastapi"}}
)
users.update_many({"age": {"$lt": 30}}, {"$inc": {"age": 1}})

# DELETE
users.delete_one({"username": "bob"})
users.delete_many({"age": {"$gt": 100}})

# Indexes
users.create_index("username", unique=True)
users.create_index([("age", 1), ("username", -1)])  # compound
users.create_index({"bio": "text"})  # text search index
'''

print(mongo_operations)

MongoDB CRUD Operations Examples

# Connect
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
db = client["ai_learning"]
users = db["users"]

# CREATE insertOne / insertMany
users.insert_one({"username": "alice", "age": 30, "tags": ["ml", "python"]})
users.insert_many([
    {"username": "bob", "age": 25},
    {"username": "carol", "age": 35, "tags": ["data"]}
])

# READ find / findOne
user = users.find_one({"username": "alice"})
all_users = list(users.find({"age": {"$gt": 25}}))
projected = list(users.find({}, {"username": 1, "_id": 0}))  # projection
sorted_users = list(users.find().sort("age", -1).limit(5))

# Query Operators:
# Comparison: $eq, $ne, $gt, $gte, $lt, $lte, $in, $nin
# Logical:    $and, $or, $nor, $not
# Element:    $exists, $type
# Array:      $all, $elemMatch, $size

# Complex query
result = users.find({
    "$and": [
        {"age": {"$gte": 25, "$lte": 40}},
        {"tags": {"$in": ["ml", "python"]}},
    ]
})

# UPDATE
users.updat

In [2]:
# MongoDB Aggregation Pipeline
aggregation_example = '''
# Aggregation pipeline like SQL GROUP BY but more powerful
pipeline = [
    # Stage 1: $match filter documents (like WHERE)
    {"$match": {"age": {"$gte": 18}}},
    
    # Stage 2: $group aggregate (like GROUP BY)
    {"$group": {
        "_id": "$country",
        "count": {"$sum": 1},
        "avg_age": {"$avg": "$age"},
        "all_tags": {"$push": "$tags"}
    }},
    
    # Stage 3: $project reshape (like SELECT)
    {"$project": {
        "country": "$_id",
        "count": 1,
        "avg_age": {"$round": ["$avg_age", 1]},
        "_id": 0
    }},
    
    # Stage 4: $sort
    {"$sort": {"count": -1}},
    
    # Stage 5: $limit
    {"$limit": 10},
    
    # $lookup like JOIN
    {"$lookup": {
        "from": "orders",
        "localField": "_id",
        "foreignField": "user_id",
        "as": "user_orders"
    }},
    
    # $unwind flatten arrays
    {"$unwind": "$user_orders"}
]

results = list(db.users.aggregate(pipeline))
'''
print(aggregation_example)


# Aggregation pipeline like SQL GROUP BY but more powerful
pipeline = [
    # Stage 1: $match filter documents (like WHERE)
    {"$match": {"age": {"$gte": 18}}},

    # Stage 2: $group aggregate (like GROUP BY)
    {"$group": {
        "_id": "$country",
        "count": {"$sum": 1},
        "avg_age": {"$avg": "$age"},
        "all_tags": {"$push": "$tags"}
    }},

    # Stage 3: $project reshape (like SELECT)
    {"$project": {
        "country": "$_id",
        "count": 1,
        "avg_age": {"$round": ["$avg_age", 1]},
        "_id": 0
    }},

    # Stage 4: $sort
    {"$sort": {"count": -1}},

    # Stage 5: $limit
    {"$limit": 10},

    # $lookup like JOIN
    {"$lookup": {
        "from": "orders",
        "localField": "_id",
        "foreignField": "user_id",
        "as": "user_orders"
    }},

    # $unwind flatten arrays
    {"$unwind": "$user_orders"}
]

results = list(db.users.aggregate(pipeline))



## Redis

Redis is an **in-memory** data structure store used as:
- Cache (reduce DB load)
- Session store
- Message broker (Pub/Sub)
- Rate limiter
- Job queue
- Real-time leaderboard

### Redis Data Structures

| Type | Commands | Use Case |
|------|----------|----------|
| **String** | SET, GET, INCR, APPEND | Cache, counters |
| **List** | LPUSH, RPUSH, LPOP, LRANGE | Queues, stacks |
| **Hash** | HSET, HGET, HGETALL | Object storage |
| **Set** | SADD, SMEMBERS, SINTER | Unique items, tags |
| **Sorted Set** | ZADD, ZRANGE, ZRANK | Leaderboards, priorities |
| **Stream** | XADD, XREAD | Event logs, ML pipelines |

In [3]:
# pip install redis
# Start Redis: docker run -d -p 6379:6379 redis

redis_examples = '''
import redis
import json
from datetime import timedelta

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

# --- STRING ---
r.set("username", "alice")
r.set("counter", 0)
r.setex("session:abc123", timedelta(hours=1), json.dumps({"user_id": 1}))  # with TTL
r.get("username")           # "alice"
r.incr("counter")           # 1 (atomic increment)
r.incrby("counter", 5)      # 6

# --- LIST (queue) ---
r.rpush("job_queue", "job1", "job2", "job3")  # right push
r.lpop("job_queue")          # "job1" (left pop = FIFO queue)
r.lrange("job_queue", 0, -1) # all items

# --- HASH (object) ---
r.hset("user:1", mapping={"name": "alice", "age": "30", "role": "admin"})
r.hget("user:1", "name")     # "alice"
r.hgetall("user:1")          # {"name": "alice", ...}
r.hincrby("user:1", "age", 1)  # age = 31

# --- SET (unique items) ---
r.sadd("user:1:tags", "ml", "python", "fastapi")
r.sadd("user:2:tags", "ml", "java", "spring")
r.smembers("user:1:tags")    # {"ml", "python", "fastapi"}
r.sinter("user:1:tags", "user:2:tags")  # {"ml"} intersection

# --- SORTED SET (leaderboard) ---
r.zadd("leaderboard", {"alice": 1500, "bob": 1200, "carol": 1800})
r.zrange("leaderboard", 0, -1, withscores=True, rev=True)  # top scores
r.zrank("leaderboard", "alice")   # rank (0-indexed)
r.zincrby("leaderboard", 100, "alice")  # alice += 100

# --- CACHING pattern ---
def get_user_with_cache(user_id: int):
    cache_key = f"user:{user_id}"
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached)  # Cache HIT
    # Cache MISS fetch from DB
    user = {"id": user_id, "name": "alice"}  # db.query(user_id)
    r.setex(cache_key, 3600, json.dumps(user))  # cache for 1 hour
    return user

# --- RATE LIMITING ---
def is_rate_limited(user_id: str, limit: int = 100, window: int = 3600) -> bool:
    key = f"rate:{user_id}"
    count = r.incr(key)
    if count == 1:
        r.expire(key, window)  # set expiry on first request
    return count > limit

# --- PUB/SUB ---
# Publisher: r.publish("ml_events", json.dumps({"event": "prediction", "result": "positive"}))
# Subscriber:
# pubsub = r.pubsub()
# pubsub.subscribe("ml_events")
# for message in pubsub.listen(): print(message)

# --- STREAM (for ML pipelines) ---
r.xadd("predictions", {"input": "text", "label": "positive", "score": "0.95"})
r.xread({"predictions": "0"}, count=10, block=1000)  # read stream
'''
print(redis_examples)


import redis
import json
from datetime import timedelta

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

# --- STRING ---
r.set("username", "alice")
r.set("counter", 0)
r.setex("session:abc123", timedelta(hours=1), json.dumps({"user_id": 1}))  # with TTL
r.get("username")           # "alice"
r.incr("counter")           # 1 (atomic increment)
r.incrby("counter", 5)      # 6

# --- LIST (queue) ---
r.rpush("job_queue", "job1", "job2", "job3")  # right push
r.lpop("job_queue")          # "job1" (left pop = FIFO queue)
r.lrange("job_queue", 0, -1) # all items

# --- HASH (object) ---
r.hset("user:1", mapping={"name": "alice", "age": "30", "role": "admin"})
r.hget("user:1", "name")     # "alice"
r.hgetall("user:1")          # {"name": "alice", ...}
r.hincrby("user:1", "age", 1)  # age = 31

# --- SET (unique items) ---
r.sadd("user:1:tags", "ml", "python", "fastapi")
r.sadd("user:2:tags", "ml", "java", "spring")
r.smembers("user:1:tags")    # {"ml", "python", "fastapi"

## Cassandra

Apache Cassandra is a **wide-column** distributed database designed for:
- High write throughput
- Linear scalability
- No single point of failure
- Time-series data, IoT, ML feature logging

### Key Concepts
- **Partition Key**: Determines which node holds the data → must be in every query
- **Clustering Key**: Orders data within a partition
- **Denormalization**: Data duplicated to avoid JOINs (no joins in Cassandra)
- **Eventual Consistency**: Writes propagate asynchronously

### CQL (Cassandra Query Language)

In [4]:
cassandra_examples = '''
# CQL Examples (run in cqlsh or via cassandra-driver)

# Create keyspace (like a database)
CREATE KEYSPACE ml_platform
WITH replication = {"class": "SimpleStrategy", "replication_factor": 3};

USE ml_platform;

# Create table optimized for query: "get predictions for user X on date Y"
CREATE TABLE predictions_by_user (
    user_id    UUID,         , partition key
    created_at TIMESTAMP,   , clustering key (ordered)
    pred_id    UUID,
    model_name TEXT,
    result     TEXT,
    confidence FLOAT,
    PRIMARY KEY ((user_id), created_at, pred_id)
) WITH CLUSTERING ORDER BY (created_at DESC);

# INSERT
INSERT INTO predictions_by_user (user_id, created_at, pred_id, model_name, result, confidence)
VALUES (uuid(), toTimestamp(now()), uuid(), "bert_v2", "positive", 0.95);

# SELECT must include partition key
SELECT * FROM predictions_by_user WHERE user_id = ?;
SELECT * FROM predictions_by_user WHERE user_id = ? AND created_at > "2024-01-01";

# Python driver
from cassandra.cluster import Cluster
cluster = Cluster(["127.0.0.1"])
session = cluster.connect("ml_platform")

prepared = session.prepare("INSERT INTO predictions_by_user ... VALUES (?, ?, ?, ?, ?, ?)")
session.execute(prepared, [user_id, created_at, pred_id, model_name, result, confidence])
'''
print(cassandra_examples)


# CQL Examples (run in cqlsh or via cassandra-driver)

# Create keyspace (like a database)
CREATE KEYSPACE ml_platform
WITH replication = {"class": "SimpleStrategy", "replication_factor": 3};

USE ml_platform;

# Create table optimized for query: "get predictions for user X on date Y"
CREATE TABLE predictions_by_user (
    user_id    UUID,         , partition key
    created_at TIMESTAMP,   , clustering key (ordered)
    pred_id    UUID,
    model_name TEXT,
    result     TEXT,
    confidence FLOAT,
    PRIMARY KEY ((user_id), created_at, pred_id)
) WITH CLUSTERING ORDER BY (created_at DESC);

# INSERT
INSERT INTO predictions_by_user (user_id, created_at, pred_id, model_name, result, confidence)
VALUES (uuid(), toTimestamp(now()), uuid(), "bert_v2", "positive", 0.95);

# SELECT must include partition key
SELECT * FROM predictions_by_user WHERE user_id = ?;
SELECT * FROM predictions_by_user WHERE user_id = ? AND created_at > "2024-01-01";

# Python driver
from cassandra.cluster import

## Choosing the Right NoSQL Database

| Use Case | Best Choice | Why |
|----------|------------|-----|
| Session/cache | Redis | In-memory, sub-millisecond |
| Product catalog | MongoDB | Flexible schema, rich queries |
| Time-series / IoT | Cassandra, InfluxDB | High write throughput |
| Social graph | Neo4j | Native graph traversal |
| Full-text search | Elasticsearch | Inverted index, relevance scoring |
| ML embeddings | Pinecone, Qdrant | ANN search |
| Real-time analytics | ClickHouse | Columnar, OLAP |
| Queue / streaming | Redis Streams, Kafka | Message passing |

## Additional Learning Resources

### Documentation
- [MongoDB Docs](https://www.mongodb.com/docs/) Complete MongoDB reference
- [Redis Docs](https://redis.io/docs/) Commands and data structures
- [Apache Cassandra Docs](https://cassandra.apache.org/doc/latest/) CQL and architecture
- [Motor Docs](https://motor.readthedocs.io/) Async MongoDB for Python

### Books
- [NoSQL Distilled](https://martinfowler.com/books/nosql.html) Martin Fowler & Pramod Sadalage
- [Designing Data-Intensive Applications](https://dataintensive.net/) Martin Kleppmann
- [Redis in Action](https://www.manning.com/books/redis-in-action) Josiah Carlson

### Papers
- [Dynamo: Amazon's Highly Available KV Store](https://www.allthingsdistributed.com/files/amazon-dynamo-sosp2007.pdf) Foundation of key-value NoSQL
- [Bigtable: A Distributed Storage System](https://research.google/pubs/pub27898/) Google, foundation of HBase/Cassandra
- [CAP Twelve Years Later](https://www.infoq.com/articles/cap-twelve-years-later-how-the-rules-have-changed/) Eric Brewer

### Courses
- [MongoDB University](https://university.mongodb.com/) Free official courses
- [Redis University](https://university.redis.com/) Free Redis courses